# HMS harmful brain activity — exploratory data analysis and preprocessing

Report sections **III. Dataset and EDA** and **IV. Preprocessing and leakage control**.

**Kaggle:** Accelerator **None** (CPU is enough) · Inputs: the **HMS competition** + the **`hms-code`** dataset ·
then **Share → Public**, and put this notebook's link in Sections III and IV of the report.
Every figure is also saved in `/kaggle/working/figures/` for the report.

In [ ]:
import os, sys, glob, json, hashlib, inspect
def find(pattern):
    hits = glob.glob('/kaggle/input/' + pattern, recursive=True)
    return os.path.dirname(hits[0]) if hits else None

CODE = os.environ.get('HMS_CODE') or find('**/hms_data.py')
sys.path.insert(0, CODE)
COMP = os.environ.get('HMS_COMP') or next((os.path.dirname(p) for p in glob.glob('/kaggle/input/**/train.csv', recursive=True)
                                           if os.path.isdir(os.path.join(os.path.dirname(p), 'train_eegs'))), None)
PREP = os.environ.get('HMS_PREP') or find('**/hms-prepared/X.npy') or '/tmp/hms-prepared'
FIGS = os.environ.get('HMS_FIGS', '/kaggle/working/figures'); os.makedirs(FIGS, exist_ok=True)
RAW = COMP is not None and os.path.isdir(os.path.join(COMP, 'train_eegs'))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import hms_data as H
pd.set_option('display.width', 160)
def save(name):
    plt.savefig(os.path.join(FIGS, name), dpi=150, bbox_inches='tight')
print('code:', CODE, '| competition:', COMP, '| raw recordings available:', RAW)

## III.1 Dataset description

In [ ]:
tr = pd.read_csv(os.path.join(COMP, 'train.csv'))
tr['total_votes'] = tr[H.VOTE_COLS].sum(1)
overview = pd.Series({
    'labelled windows (rows of train.csv)': len(tr),
    'recordings (eeg_id)': tr.eeg_id.nunique(),
    'patients (patient_id)': tr.patient_id.nunique(),
    'labelled windows per recording (median / max)': '%d / %d' % (tr.groupby('eeg_id').size().median(), tr.groupby('eeg_id').size().max()),
    'recordings per patient (median / max)': '%d / %d' % (tr.groupby('patient_id').eeg_id.nunique().median(), tr.groupby('patient_id').eeg_id.nunique().max()),
    'expert votes per window (median / max)': '%d / %d' % (tr.total_votes.median(), tr.total_votes.max()),
})
print(overview.to_string())
print('\ncolumns of train.csv:', list(tr.columns))
demo = [c for c in tr.columns if c.lower() in ('age', 'sex', 'gender')]
print('demographic columns:', demo or 'NONE: age and sex are not provided, so bias by age or sex cannot be assessed')

In [ ]:
if RAW:
    import pyarrow.parquet as pq
    f0 = os.path.join(COMP, 'train_eegs', '%d.parquet' % tr.eeg_id.iloc[0])
    cols = list(pd.read_parquet(f0).columns)
    print(len(cols), 'channels:', cols, '| sampling rate %d Hz' % H.FS_RAW)
    ids = tr.eeg_id.unique()
    dur = np.array([pq.ParquetFile(os.path.join(COMP, 'train_eegs', '%d.parquet' % i)).metadata.num_rows for i in ids]) / H.FS_RAW
    print('recording length (s): min %.0f | median %.0f | 95th percentile %.0f | max %.0f' % (dur.min(), np.median(dur), np.percentile(dur, 95), dur.max()))
    plt.figure(figsize=(6, 3.2)); plt.hist(dur / 60, bins=80)
    plt.xlabel('recording length (minutes)'); plt.ylabel('recordings'); plt.yscale('log'); save('fig_recording_length.png'); plt.show()
else:
    print('raw recordings not attached: channel list and recording lengths skipped')

### Prepared data (one window per recording, see Section IV)

In [ ]:
if not os.path.exists(os.path.join(PREP, 'X.npy')):
    H.prepare(COMP, PREP)
data = H.load(PREP); meta = data.meta
Y = H.soft_targets(meta); maj = np.array(H.CLASSES)[Y.argmax(1)]
print('X', data.X.shape, '(windows, samples at %d Hz, bipolar channels) | F' % H.FS_MODEL, data.F.shape)

## III.2 Class imbalance, at window level and at patient level

In [ ]:
maj_all = pd.Series(np.array(H.CLASSES)[tr[H.VOTE_COLS].values.argmax(1)])
t = tr.assign(m=maj_all.values)
pat_any = t.groupby('m').patient_id.nunique()
pat_major = t.groupby('patient_id').m.agg(lambda s: s.value_counts().idxmax()).value_counts()
bal = pd.DataFrame({'windows, all 106,800': maj_all.value_counts(),
                    'windows, 1 per recording': pd.Series(maj).value_counts(),
                    'patients with >=1 window': pat_any,
                    'patients, majority class': pat_major}).reindex(H.CLASSES).fillna(0).astype(int)
ratio = (bal.max() / bal.replace(0, np.nan).min()).round(1)
display(bal); print('imbalance ratio (largest / smallest class):'); print(ratio.to_string())
ax = (bal / bal.sum()).plot.bar(figsize=(7, 3.2), rot=0); ax.set_ylabel('share'); ax.set_title('Majority label per class')
save('fig_class_balance.png'); plt.show()

## III.2 Label quality: number of votes and expert disagreement

In [ ]:
tv = meta.total_votes.values
print('votes per window: median %d | 1 vote: %.1f%% | >=10 votes: %.1f%%' % (np.median(tv), 100 * (tv == 1).mean(), 100 * (tv >= 10).mean()))
print('unanimous windows (all votes on one class): %.1f%%' % (100 * (Y.max(1) == 1).mean()))
groups = {'1 vote': (1, 1), '2 votes': (2, 2), '3-5 votes': (3, 5), '6-9 votes': (6, 9), '>=10 votes': (10, 999)}
comp = pd.DataFrame({g: pd.Series(maj[(tv >= a) & (tv <= b)]).value_counts(normalize=True) for g, (a, b) in groups.items()}).reindex(H.CLASSES).fillna(0).round(3)
comp.loc['windows'] = [int(((tv >= a) & (tv <= b)).sum()) for a, b in groups.values()]
display(comp)
fig, ax = plt.subplots(figsize=(6, 3.2)); ax.hist(np.minimum(tv, 30), bins=30)
ax.set_xlabel('expert votes per window (30 = 30 or more)'); ax.set_ylabel('windows'); save('fig_votes.png'); plt.show()

## III.1 One representative window per class (the window with the highest agreement)

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(20, 6), sharey=True)
t_ax = np.arange(data.X.shape[1]) / H.FS_MODEL
for ax, c in zip(axes, H.CLASSES):
    k = np.flatnonzero(maj == c)
    i = k[np.lexsort((-tv[k], -Y[k].max(1)))[0]]
    x = np.asarray(data.X[i], np.float32)
    for ch in range(x.shape[1]):
        ax.plot(t_ax, x[:, ch] / (np.abs(x[:, ch]).max() + 1e-9) * 0.45 - ch, lw=0.5, color='k')
    ax.set_title('%s (%d%% of %d votes)' % (c.upper(), round(100 * Y[i].max()), tv[i]), fontsize=10)
    ax.set_xlabel('s')
axes[0].set_yticks(-np.arange(x.shape[1])); axes[0].set_yticklabels(H.CHANNELS, fontsize=7)
plt.tight_layout(); save('fig_examples_per_class.png'); plt.show()

## III.1 Signal statistics (domain equivalent of a correlation matrix)

In [ ]:
sd = np.concatenate([np.asarray(data.X[s:s + 2000], np.float32).std(1) for s in range(0, len(meta), 2000)])
amp = pd.DataFrame(sd, columns=H.CHANNELS).describe(percentiles=[.05, .5, .95]).T[['5%', '50%', '95%']].round(1)
print('standard deviation of each channel within a window (microvolts, after filtering):'); display(amp.T)
nb_ = len(H.BANDS); rel = data.F[:, 16 * nb_:].reshape(len(meta), 16, nb_).mean(1)
logp = data.F[:, :16 * nb_].reshape(len(meta), 16, nb_).mean(1)
bands = [b for b, _, _ in H.BANDS]
heat = pd.DataFrame(rel, columns=bands).groupby(maj).mean().reindex(H.CLASSES)
corr = pd.DataFrame(logp, columns=bands).corr()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
im = a1.imshow(heat.values, cmap='viridis'); a1.set_xticks(range(nb_)); a1.set_xticklabels(bands); a1.set_yticks(range(6)); a1.set_yticklabels(H.CLASSES)
a1.set_title('relative band power by class'); plt.colorbar(im, ax=a1)
im = a2.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1); a2.set_xticks(range(nb_)); a2.set_xticklabels(bands); a2.set_yticks(range(nb_)); a2.set_yticklabels(bands)
a2.set_title('correlation of log band power'); plt.colorbar(im, ax=a2)
plt.tight_layout(); save('fig_band_power.png'); plt.show()
display(heat.round(3))

## III.2 Missing data, noise and duplicates

In [ ]:
print('windows with a missing electrode (NaN): %.1f%% | missing samples overall: %.3f%%'
      % (100 * (meta.missing_frac > 0).mean(), 100 * meta.missing_frac.mean()))
flat = (sd < 1e-3).any(1); big = sd.max(1) > 200
print('windows with at least one flat channel: %d (%.1f%%)' % (flat.sum(), 100 * flat.mean()))
print('windows with a channel above 200 microvolts of standard deviation (artifact-like): %d (%.1f%%)' % (big.sum(), 100 * big.mean()))
rows = tr.groupby('eeg_id').size()
ov = 0
for _, g in tr.groupby('eeg_id'):
    o = np.sort(g.eeg_label_offset_seconds.values)
    ov += int((np.diff(o) < H.WIN_S).sum())
print('labelled windows per recording: %.1f on average; %d consecutive pairs overlap in time (same recording, offsets < %d s apart)' % (rows.mean(), ov, H.WIN_S))
print('-> near-duplicates inside a recording: the prepared data keep ONE window per recording')
allflat = (sd < 1e-3).all(1)
print('windows entirely flat (all 16 channels constant, no usable signal): %d %s'
      % (allflat.sum(), dict(pd.Series(meta.split[allflat]).value_counts())))
h = pd.Series([hashlib.md5(np.asarray(data.X[i]).tobytes()).hexdigest() for i in range(len(meta))])
print('identical windows across different recordings, flat ones excluded:', int((h.duplicated(keep=False) & ~allflat).sum()))

In [ ]:
if RAW:
    rng = np.random.default_rng(0); ids_all = tr.eeg_id.unique(); sample = rng.choice(ids_all, min(300, len(ids_all)), replace=False)
    nan_frac = pd.DataFrame([pd.read_parquet(os.path.join(COMP, 'train_eegs', '%d.parquet' % i))[H.ELECTRODES].isna().mean() for i in sample])
    print('share of missing samples per electrode, %d random recordings:' % len(sample)); display(nan_frac.mean().round(4).to_frame('missing').T)
else:
    print('raw recordings not attached: per-electrode missingness skipped')

## IV.1 Domain-specific preprocessing

All in `hms_data.py` (`prepare`). Per recording: one labelled window → 16 bipolar chains (double banana) →
centre each chain on its median, missing samples set to 0, clip at ±1024 µV → 60 Hz notch + 0.5–45 Hz
band-pass (zero-phase) on the full 50 s → resample to 100 Hz → keep the central 10 s → `X` (1000 × 16).
Band power in 5 bands per chain → `F` (160 features).

In [ ]:
print({k: getattr(H, k) for k in ('FS_RAW', 'FS_MODEL', 'WIN_S', 'LABEL_S', 'CLIP_UV', 'NOTCH_HZ', 'BANDPASS')})
print('bands:', H.BANDS); print('bipolar chains:', H.CHANNELS)
if RAW:
    r0 = tr.iloc[0]
    raw = H.read_window(os.path.join(COMP, 'train_eegs', '%d.parquet' % r0.eeg_id), r0.eeg_label_offset_seconds)
    bip, _ = H.to_bipolar(raw); filt = H.filter_signal(bip)
    t50 = np.arange(len(raw)) / H.FS_RAW
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
    a1.plot(t50, raw[:, H.ELECTRODES.index('Fp1')], lw=0.5); a1.set_title('raw electrode Fp1 (referential, 200 Hz)')
    a2.plot(t50, filt[:, 0], lw=0.5); a2.set_title('chain Fp1-F7 after notch + 0.5-45 Hz band-pass')
    for a in (a1, a2): a.axvspan(20, 30, color='orange', alpha=0.2)
    a2.set_xlabel('s (shaded: the labelled central 10 s kept)'); plt.tight_layout(); save('fig_preprocessing.png'); plt.show()

## IV.2 Zero data leakage

**Split by patient.** In `prepare`, the 1,950 patients are dealt into 10 groups of similar size
(`assign_split`): group 0 = test, group 1 = validation, groups 2–9 = training pool. Done once, before any
model sees the data. **Fit on training data only:** the only data-dependent step, the per-channel scaling,
is computed in `make_split` from the training windows and then applied to validation and test (source
printed below). No oversampling and no augmentation are used.

In [ ]:
m = meta
train_ids, val_ids, test_ids = set(m.patient_id[m.split == 'train']), set(m.patient_id[m.split == 'val']), set(m.patient_id[m.split == 'test'])
assert train_ids.isdisjoint(test_ids), "DATA LEAKAGE DETECTED: Train/Test overlap!"
assert train_ids.isdisjoint(val_ids), "DATA LEAKAGE DETECTED: Train/Val overlap!"
assert val_ids.isdisjoint(test_ids), "DATA LEAKAGE DETECTED: Val/Test overlap!"
rec = {s: set(m.eeg_id[m.split == s]) for s in ('train', 'val', 'test')}
assert rec['train'].isdisjoint(rec['test']) and rec['train'].isdisjoint(rec['val']) and rec['val'].isdisjoint(rec['test'])
print('no patient and no recording in two sets | patients: train %d, val %d, test %d' % (len(train_ids), len(val_ids), len(test_ids)))
display(pd.DataFrame(H.describe_split(m)).T)
display(pd.crosstab(m.split, maj, normalize='index').reindex(columns=H.CLASSES).round(3))
src = inspect.getsource(H.make_split).split('\n')
print('\n'.join(l for l in src if any(k in l for k in ('xtr =', 'pick =', 'med =', 'q75, q25', 'iqr =', 'norm =', 's.X_train, s.X_val'))))

In [ ]:
print('figures for the report:', sorted(os.listdir(FIGS)))